# Rami-AI — joue au Rami contre une IA qui voit les cartes

Pose ton iPad au-dessus de la table, joue au Rami contre une IA qui voit les cartes via la caméra. 3 niveaux : Découverte, Stratégie, Champion (RL).

**Auteur :** Amine Harch El Korane  
**Licence :** MIT  
**Code :** https://github.com/Vitalcheffe/ramai-ai  
**Tests :** 131 passent

---

## Mode d'utilisation

Le notebook fonctionne en **deux modes** :

- **MANUEL** (par défaut) : tu saisis les cartes (main, défausse) par grille cliquable. Aucune caméra nécessaire. Marche du premier coup, sans entraîner de modèle.
- **AUTO** : la caméra détecte les cartes automatiquement. Nécessite un modèle YOLO pré-entraîné (téléchargé automatiquement si disponible, sinon entraînement dans Colab).

Le mode MANUEL est le plus rapide pour tester. Le mode AUTO est le "vrai" projet.

---

## Setup physique

Sur ta table :
- **Tes 14 cartes** : en main (face vers toi, dos vers la caméra)
- **Le talon (stock)** : face cachée, au centre
- **La défausse** : face visible, à côté du talon
- **Melds posées** : face visible, sur la table

La caméra (si mode AUTO) voit : défausse + melds posées.  
La caméra ne voit PAS : ta main, le talon.  
→ Ta main est saisie manuellement (grille cliquable).

---

Exécute les cellules dans l'ordre.

## Cellule 1 — Installation, imports, et préchauffage caméra

Cette cellule installe les deps, importe le code, ET précharge la caméra (pour éviter le bug "la permission n'apparaît jamais").

In [ ]:
# Installe les dépendances
!pip install -q ultralytics ipywidgets 2>&1 | tail -3

import os, sys, json, time, random
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage, clear_output
import ipywidgets as widgets
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Clone le repo si pas déjà présent
if not Path('/content/ramai-ai').exists():
    !git clone -q https://github.com/Vitalcheffe/ramai-ai.git /content/ramai-ai
    
REPO = Path('/content/ramai-ai')
sys.path.insert(0, str(REPO))

# Importe le moteur de jeu + les IA + le protocole + vision
from rami.config import RamiConfig
from rami.cards import Card, build_deck, Hand, SUIT_SYMBOLS, RANK_NAMES
from rami.engine import (is_valid_meld, valid_melds, deadwood_score,
                          best_meld_partition, meld_points)
from rami.game import new_game, legal_moves, apply_move, Move, GameState
from rami.extensions import (designate_jokers, find_meld_extensions,
                              all_laid_melds, JokerDesignation)
from rami.counting import CardCountingState
from rami.protocol import (
    ProtocolStep, TurnContext, next_step,
    should_show_ai_hand, show_ai_hand_warning,
)
from rami.ai.discovery import DiscoveryAI
from rami.ai.strategy import StrategyAI
from rami.ai.champion import ChampionAI
from rami.vision import (
    CardDetector, MockDetector, calibrate_camera,
    detect_discard_pile, detect_meld_clusters, find_extendable_melds,
    prewarm_camera, capture_photo, is_camera_ready, get_camera_status,
    try_download_pretrained, list_candidate_urls, get_model_info,
)

print('✓ Rami-AI chargé')
print(f'  Tests : 131 (dont 12 sur camera + mode manuel)')
print()
print('--- Préchauffage caméra ---')
print('Une popup "Autoriser la caméra" devrait apparaître dans le navigateur.')
print('Clique sur Autoriser. Si rien ne s\'affiche, le mode MANUEL sera utilisé.')
print()
cam_status = prewarm_camera()
print(f'Caméra : {cam_status}')
if cam_status == 'granted':
    print('  ✓ Caméra prête. Tu pourras faire des photos plus tard.')
    globals()['CAMERA_READY'] = True
else:
    print('  ✗ Caméra non disponible. Le notebook passera en mode MANUEL.')
    globals()['CAMERA_READY'] = False

## Cellule 2 — Choix du mode (MANUEL ou AUTO) + variante

In [ ]:
mode = widgets.Dropdown(
    options=[('MANUEL — saisie par grille (recommandé)', 'manuel'),
             ('AUTO — détection caméra (si YOLO dispo)', 'auto')],
    value='manuel',
    description='Mode :',
    style={'description_width': 'initial'}
)
ai_level = widgets.Dropdown(
    options=[('Découverte (règles pures, main IA visible)', 'discovery'),
             ('Stratégie (comptage parfait, main IA cachée)', 'strategy'),
             ('Champion (RL self-play, main IA cachée)', 'champion')],
    value='strategy',
    description='Niveau IA :',
    style={'description_width': 'initial'}
)
variant = widgets.Dropdown(
    options=[('Marocain classique (seuil 30)', 'classic'),
             ('Rami 51 (seuil 51, pas de défausse avant seuil)', '51'),
             ('Sans seuil', 'none'),
             ('Sans jokers', 'nojokers')],
    value='classic',
    description='Variante :',
    style={'description_width': 'initial'}
)
display(mode, ai_level, variant)

confirm = widgets.Button(description='Valider la configuration', button_style='primary')
output_area = widgets.Output()
display(confirm, output_area)

def on_confirm(b):
    output_area.clear_output()
    
    # Mode AUTO requires camera + YOLO
    use_camera = (mode.value == 'auto')
    if use_camera and not globals().get('CAMERA_READY', False):
        with output_area:
            print('⚠ Caméra non disponible — passage en mode MANUEL.')
        use_camera = False
    
    if variant.value == 'classic':
        cfg = RamiConfig.classic_moroccan()
    elif variant.value == '51':
        cfg = RamiConfig.threshold_51()
    elif variant.value == 'none':
        cfg = RamiConfig.no_threshold()
    else:
        cfg = RamiConfig.no_jokers()
    
    if ai_level.value == 'discovery':
        ai = DiscoveryAI(seed=0)
    elif ai_level.value == 'strategy':
        ai = StrategyAI(seed=0)
    else:
        weights_path = str(REPO / 'models' / 'champion_weights.json')
        if not os.path.exists(weights_path):
            with output_area:
                print('⚠ Champion pas encore entraîné. Entraînement rapide (500 parties)...')
                !cd {REPO} && python scripts/train_champion.py --games 500 --candidates 6
        ai = ChampionAI(weights_path=weights_path, seed=0)
    
    # Try to load YOLO if AUTO mode
    detector = MockDetector()  # default
    if use_camera:
        weights = REPO / 'models' / 'yolo_cards.pt'
        if not weights.exists():
            with output_area:
                print('Téléchargement d\'un modèle YOLO pré-entraîné...')
                print('(Essai de plusieurs URLs publiques)')
                result = try_download_pretrained(out_path=str(weights), timeout=5)
                if result is None:
                    print('⚠ Aucun modèle pré-entraîné trouvé. Pour entraîner:')
                    print(f'  !cd {REPO} && python scripts/train_yolo.py --data cards.yaml --epochs 50')
                    print('Passage en mode MANUEL.')
                    use_camera = False
                else:
                    info = get_model_info(str(weights))
                    print(f'  ✓ Modèle téléchargé: {info["size_mb"]} MB')
        if weights.exists():
            try:
                detector = CardDetector(weights_path=str(weights))
                print(f'✓ Modèle YOLO chargé')
            except Exception as e:
                print(f'⚠ Erreur chargement YOLO: {e}')
                use_camera = False
    
    with output_area:
        print()
        print(f'✓ Configuration validée')
        print(f'  Mode : {"AUTO (caméra)" if use_camera else "MANUEL (grille)"}')
        print(f'  IA : {ai.name}')
        print(f'  Variante : {variant.label}')
        print(f'  Seuil première pose : {cfg.first_meld_threshold} pts')
        if cfg.block_discard_before_threshold:
            print(f'  Rami 51 : PAS de prise défausse avant seuil')
        print(f'  Main IA visible : {"oui" if should_show_ai_hand(ai_level.value) else "non (cachée)"}')
        globals()['CFG'] = cfg
        globals()['AI'] = ai
        globals()['AI_LEVEL'] = ai_level.value
        globals()['USE_CAMERA'] = use_camera
        globals()['DETECTOR'] = detector

confirm.on_click(on_confirm)

## Cellule 3 — Saisie manuelle de ta main (14 cartes)

Clique sur les 14 cartes que tu as en main. Pas de caméra pour ça — l'iPad voit le dos de tes cartes.

(Le tableau ci-dessous contient les 52 cartes + 2 jokers, cliquables.)

In [ ]:
def make_card_button(rank, suit):
    if rank == 0:
        label = '★'
        color = 'purple'
    else:
        label = f'{RANK_NAMES[rank]}{SUIT_SYMBOLS[suit]}'
        color = 'red' if suit in (1, 2) else 'black'  # ♥♦ = red, ♠♣ = black
    btn = widgets.ToggleButton(
        description=label,
        button_style='',
        layout=widgets.Layout(width='50px', height='40px'),
        style={'button_color': 'white', 'font_weight': 'bold'},
    )
    btn.rank = rank
    btn.suit = suit
    btn.color = color
    return btn

def make_hand_selector():
    grid = widgets.GridBox(
        layout=widgets.Layout(
            grid_template_columns='repeat(13, 50px)',
            grid_gap='4px'
        )
    )
    buttons = []
    for suit in range(4):
        for rank in range(1, 14):
            btn = make_card_button(rank, suit)
            buttons.append(btn)
    for j in range(2):
        btn = make_card_button(0, -1)
        buttons.append(btn)
    grid.children = buttons
    return grid, buttons

selected_count = widgets.Label(value='Cartes sélectionnées : 0 / 14')
grid, all_buttons = make_hand_selector()
display(widgets.HTML('<b>Ta main (clique les 14 cartes que tu as) :</b>'))
display(selected_count, grid)

def update_count():
    n = sum(1 for b in all_buttons if b.value)
    selected_count.value = f'Cartes sélectionnées : {n} / 14'

for b in all_buttons:
    b.observe(lambda c: update_count(), 'value')

validate_hand_btn = widgets.Button(description='Valider ma main', button_style='success')
hand_out = widgets.Output()
display(validate_hand_btn, hand_out)

def on_validate_hand(b):
    hand_out.clear_output()
    selected = [Card(suit=b.suit, rank=b.rank, copy_id=0) for b in all_buttons if b.value]
    if len(selected) != 14:
        with hand_out:
            print(f'⚠ Tu as sélectionné {len(selected)} cartes. Il en faut exactement 14.')
        return
    globals()['HUMAN_HAND'] = selected
    with hand_out:
        print(f'✓ Main validée : {" ".join(c.name for c in selected)}')

validate_hand_btn.on_click(on_validate_hand)

## Cellule 4 — Initialisation de la partie

In [ ]:
CFG = globals().get('CFG', RamiConfig())
AI = globals().get('AI', StrategyAI(seed=0))
AI_LEVEL = globals().get('AI_LEVEL', 'strategy')
USE_CAMERA = globals().get('USE_CAMERA', False)
DETECTOR = globals().get('DETECTOR', MockDetector())

state = new_game(CFG, seed=int(time.time()) % 1000)
if 'HUMAN_HAND' in globals():
    state.players[0].hand.cards = list(globals()['HUMAN_HAND'])
counting = CardCountingState.fresh(
    CFG, ai_player_idx=1,
    ai_hand=state.players[1].hand.cards,
    initial_discard=state.discard,
)

print(f"Partie démarrée.")
print(f"  Toi (P0) : 14 cartes en main (cachées)")
print(f"  RAMAI (P1, {AI.name}) : {len(state.players[1].hand)} cartes en main")
print(f"  Mode : {'AUTO (caméra)' if USE_CAMERA else 'MANUEL (grille)'}")
print(f"  Top de la défausse : {state.top_discard.name if state.top_discard else '—'}")
print(f"  Stock : {len(state.stock)} cartes")
if should_show_ai_hand(AI_LEVEL):
    print(f"  Main RAMAI : {' '.join(c.name for c in state.players[1].hand.cards)}")
    print(f"    (visible — mode Découverte)")

## Cellule 5 — Tour de l'humain

1. Choisis ta source de pioche (talon ou défausse)
2. Si tu poses une meld : sélectionne les cartes dans la grille ci-dessous
3. Choisis ta carte de défausse
4. Termine ton tour (photo défausse en mode AUTO, ou saisie manuelle)

In [ ]:
draw_stock_btn = widgets.Button(description='Pioche talon', button_style='info')
draw_discard_btn = widgets.Button(description='Prends défausse', button_style='warning')
if CFG.block_discard_before_threshold and not state.players[0].has_laid_first:
    draw_discard_btn.disabled = True
    draw_discard_btn.tooltip = 'Rami 51 : interdit avant le seuil'

end_turn_btn = widgets.Button(description='Termine mon tour', button_style='success')
end_turn_btn.disabled = True

human_out = widgets.Output()
display(widgets.HBox([draw_stock_btn, draw_discard_btn]), end_turn_btn, human_out)

human_action = {'draw_source': None, 'discard_card': None, 'photo_taken': False}

def on_draw_stock(b):
    human_out.clear_output()
    if not state.stock:
        with human_out: print('Talon vide')
        return
    drawn = state.stock.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'stock', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'stock'
    with human_out:
        print(f'Tu as pioché : {drawn.name}')
        print(f'Ta main : {len(state.players[0].hand)} cartes (cachées)')
        print('Maintenant : choisis quelle carte jeter.')
        print('(Sélectionne-la dans la grille ci-dessous.)')

def on_draw_discard(b):
    human_out.clear_output()
    if not state.discard:
        with human_out: print('Pas de défausse')
        return
    drawn = state.discard.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'discard', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'discard'
    with human_out:
        print(f'Tu as pris la défausse : {drawn.name}')
        print(f'Ta main : {len(state.players[0].hand)} cartes (cachées)')
        print('Maintenant : choisis quelle carte jeter.')

def on_end_turn(b):
    human_out.clear_output()
    if not human_action['draw_source']:
        with human_out: print('Tu dois d\'abord piocher.')
        return
    # Check discard card selected
    discard_card = None
    selected_for_discard = [b for b in all_buttons if b.value and b not in globals().get('_hand_validated_buttons', set())]
    # For simplicity, ask user to select discard in a separate step
    if not human_action.get('discard_card'):
        with human_out:
            print('⚠ Sélectionne la carte à jeter dans la grille ci-dessus, puis clique à nouveau.')
            # Get the latest selected card as discard
            latest = None
            for btn in all_buttons:
                if btn.value:
                    # Skip cards that were in original hand
                    if 'HUMAN_HAND' in globals():
                        original_keys = {(c.suit, c.rank) for c in globals()['HUMAN_HAND']}
                        if (btn.suit, btn.rank) in original_keys and len([x for x in all_buttons if x.value]) == 14:
                            continue
                    latest = btn
            if latest:
                human_action['discard_card'] = Card(suit=latest.suit, rank=latest.rank, copy_id=0)
                print(f'Carte à jeter : {human_action["discard_card"].name}')
                print('Clique encore sur "Termine mon tour" pour confirmer.')
        return
    # Apply discard
    discard = human_action['discard_card']
    state.players[0].hand.remove(discard)
    state.discard.append(discard)
    counting.record_discard(0, discard)
    human_action['photo_taken'] = True  # manual mode = always OK
    
    with human_out:
        est = counting.opponent_hand_estimate(CFG, opponent_idx=0,
                                                stock_size=len(state.stock))
        print(f'Ta main déduite par RAMAI : {est["hand_count"]} cartes')
        print(f'  (arithmétique : {"OK" if est["arithmetic_consistent"] else "INCOHÉRENT"})')
        if counting.is_opponent_empty(0):
            print('🏁 Tu as gagné !')
            state.winner = 0
            state.terminal = True
            return
        state.current = 1
        state.turn += 1
        print(f"→ Tour de RAMAI (Cellule 6)")

draw_stock_btn.on_click(on_draw_stock)
draw_discard_btn.on_click(on_draw_discard)
end_turn_btn.on_click(on_end_turn)

## Cellule 6 — Tour de RAMAI

RAMAI annonce sa décision, l'explique en français (y compris quelle carte chaque joker remplace), et joue. Tu exécutes physiquement le coup si tu veux.

In [ ]:
ai_play_btn = widgets.Button(description='🎯 RAMAI joue', button_style='primary')
ai_out = widgets.Output()
display(ai_play_btn, ai_out)

def explain_move(state, move, ai):
    lines = []
    if move.draw_source == 'discard':
        top = state.top_discard
        lines.append(f"Je prends la défausse ({top.name}).")
    else:
        lines.append(f"Je pioche dans le talon (carte inconnue).")
    if move.laydowns:
        lines.append(f"Je pose {len(move.laydowns)} meld(s) :")
        for meld in move.laydowns:
            cards_str = ' '.join(c.name for c in meld)
            lines.append(f"  → {cards_str}")
            desigs = designate_jokers(meld, CFG)
            for d in desigs:
                lines.append(f"      {d.name}")
    else:
        lines.append(f"Je ne pose rien ce tour.")
    lines.append(f"Je jette : {move.discard.name}")
    return '\n'.join(lines)

def on_ai_play(b):
    ai_out.clear_output()
    with ai_out:
        if state.terminal:
            print('Partie terminée.')
            return
        if state.current != 1:
            print("Ce n'est pas le tour de RAMAI. Joue d'abord ton tour (Cellule 5).")
            return
        print(f'--- Tour {state.turn + 1} | RAMAI ({AI.name}) ---')
        if should_show_ai_hand(AI_LEVEL):
            print(f'Main RAMAI : {" ".join(c.name for c in state.players[1].hand.cards)}')
        print()
        
        m = AI.decide(state)
        print(explain_move(state, m, AI))
        
        # Apply the move
        if m.draw_source == 'stock':
            drawn = state.stock.pop()
            counting.record_draw(1, 'stock', drawn, ai_player_idx=1)
        else:
            drawn = state.discard.pop()
            counting.record_draw(1, 'discard', drawn, ai_player_idx=1)
        state.players[1].hand.add(drawn)
        
        for meld in m.laydowns:
            for card in meld:
                state.players[1].hand.remove(card)
            state.players[1].laid_melds.append(meld)
            counting.record_meld(1, meld)
            if not state.players[1].has_laid_first:
                state.players[1].has_laid_first = True
        
        state.players[1].hand.remove(m.discard)
        state.discard.append(m.discard)
        counting.record_discard(1, m.discard)
        
        print()
        print(f'Top de la défausse : {state.top_discard.name if state.top_discard else "—"}')
        
        if counting.is_opponent_empty(1):
            print('🏁 RAMAI a gagné !')
            state.winner = 1
            state.terminal = True
        else:
            state.current = 0
            state.turn += 1
            print()
            print(f"→ À toi (P0). Top défausse : {state.top_discard.name}")

ai_play_btn.on_click(on_ai_play)

## Cellule 7 — Bouton "triche" : voir la main de RAMAI

Par défaut, la main de RAMAI est cachée (sauf mode Découverte). Ce bouton te permet de la révéler — avec un warning.

In [ ]:
triche_btn = widgets.Button(description='👁 Triche : voir main RAMAI', button_style='danger')
triche_out = widgets.Output()
display(triche_btn, triche_out)

triche_confirmed = [False]

def on_triche(b):
    triche_out.clear_output()
    if not triche_confirmed[0]:
        triche_confirmed[0] = True
        with triche_out:
            print(show_ai_hand_warning())
            print('Clique encore une fois pour confirmer.')
        return
    triche_confirmed[0] = False
    with triche_out:
        if state.terminal:
            print('Partie terminée.')
            return
        print('⚠ TRICHE — Main de RAMAI :')
        print('  ', ' '.join(c.name for c in state.players[1].hand.cards))
        print()
        est = counting.opponent_hand_estimate(CFG, opponent_idx=1,
                                                stock_size=len(state.stock))
        print(f'Card counting :')
        print(f'  Main RAMAI déduite : {est["hand_count"]} cartes')
        print(f'  Main RAMAI réelle  : {len(state.players[1].hand)} cartes')

triche_btn.on_click(on_triche)

## Cellule 8 — Extensions de melds + désignation des jokers

Quand des melds sont posées sur la table, RAMAI détecte si tu (ou elle) pouvez étendre une meld existante.

In [ ]:
ext_out = widgets.Output()
display(ext_out)

with ext_out:
    print('=== Melds posées sur la table ===')
    print()
    all_melds = all_laid_melds(state)
    if not all_melds:
        print('Aucune meld posée pour l\'instant.')
    else:
        for p_idx, m_idx, meld in all_melds:
            player_name = 'Toi' if p_idx == 0 else 'RAMAI'
            print(f'Meld de {player_name} : {" ".join(c.name for c in meld)}')
            desigs = designate_jokers(meld, CFG)
            for d in desigs:
                print(f'  {d.name}')
            print()
    
    ai_hand = state.players[1].hand.cards
    extensions_found = []
    for card in ai_hand:
        if card.is_joker:
            continue
        exts = find_meld_extensions(card, all_melds, CFG)
        for ext in exts:
            extensions_found.append((card, ext))
    if extensions_found:
        print('Extensions possibles pour RAMAI :')
        for card, ext in extensions_found:
            target = 'Toi' if ext.meld_owner == 0 else 'RAMAI'
            print(f'  {card.name} → étend meld {ext.meld_index} de {target} ({ext.extends_at})')
    else:
        print('Aucune extension possible pour RAMAI.')

## Cellule 9 — Calibration caméra (mode AUTO uniquement)

Si tu es en mode AUTO, cette cellule vérifie que la caméra est bien cadrée et que l'angle est acceptable. Sinon, elle affiche "mode MANUEL".

In [ ]:
calib_out = widgets.Output()
display(calib_out)

if not USE_CAMERA:
    with calib_out:
        print('Mode MANUEL — pas de calibration caméra nécessaire.')
        print('Pour activer le mode AUTO :')
        print('  1. Re-exécute la Cellule 2 en choisissant "AUTO"')
        print('  2. La caméra doit être autorisée (Cellule 1)')
        print('  3. Un modèle YOLO doit être disponible')
else:
    calibrate_btn = widgets.Button(description='📸 Calibration', button_style='primary')
    display(calibrate_btn)
    
    def on_calibrate(b):
        calib_out.clear_output()
        with calib_out:
            print('Capture...')
            img = capture_photo()
            if img is None:
                print('Erreur : image vide. Vérifie que la caméra est autorisée.')
                return
            result = calibrate_camera(img)
            print(f'Angle : {result.tilt_degrees:.1f}°')
            print(result.message)
            # Draw frame
            color = (0, 255, 0) if result.is_good else (0, 0, 255)
            h, w = img.shape[:2]
            margin = 50
            cv2.rectangle(img, (margin, margin), (w - margin, h - margin), color, 4)
            from google.colab.patches import cv2_imshow
            cv2_imshow(img)
    
    calibrate_btn.on_click(on_calibrate)

## Cellule 10 — Fin de partie : analyse

In [ ]:
print('=' * 60)
print('ANALYSE DE LA PARTIE')
print('=' * 60)
print()

if state.winner is not None:
    winner = 'Toi' if state.winner == 0 else 'RAMAI'
    print(f'Gagnant : {winner}')
else:
    print('Partie non terminée. Continue (Cellules 5-6).')

print(f'Tours joués : {state.turn}')
print()
print('Melds posés par toi :')
for i, m in enumerate(state.players[0].laid_melds):
    print(f'  {i+1}. {" ".join(c.name for c in m)}')
print()
print('Melds posés par RAMAI :')
for i, m in enumerate(state.players[1].laid_melds):
    print(f'  {i+1}. {" ".join(c.name for c in m)}')
    desigs = designate_jokers(m, CFG)
    for d in desigs:
        print(f'     {d.name}')

print()
print('Card counting final :')
for p in range(CFG.num_players):
    name = 'Toi' if p == 0 else 'RAMAI'
    h = counting.hand_count(p)
    print(f'  Main {name} (déduite) : {h} cartes')

## Cellule bonus — Vérification des 131 tests

Exécute les tests pour vérifier que tout marche.

In [ ]:
!cd {REPO} && python -m pytest tests/ 2>&1 | tail -5